# This notebook converts the .mat data to netcdf format, 
# add the necessary metadata and plot the data

## load library

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os, sys
import json
# Import your own modules
sys.path.append('/Users/yugao/UOP/ORS-processing')
from src.netcdf import read_mat_file, read_metadata_json, save_preserved_info_from_json, mat_to_xarray, save_to_netcdf

## read mat file

In [2]:
ls ../../data/external/

OS_NTAS_2016_D_TS.json    OS_NTAS_2016_D_TS.txt     stratus12_sbe16_1879.mat*
OS_NTAS_2016_D_TS.nc      stratus12_sbe16_1876.mat*


In [3]:
mat_filepath = '../../data/external/stratus12_sbe16_1876.mat'
mat_data = read_mat_file(mat_filepath)
# Inspect the structure or contents of mat_data
print(mat_data.keys())  # Adjust based on your data

dict_keys(['__header__', '__version__', '__globals__', 'meta', 'latitude', 'longitude', 'depth', 'mday', 'temp', 'cond', 'sal', 'sal_sbe', 'year', 'yday'])


## convert mat file to netcdf

In [4]:
sbe16_metadata = mat_data['meta']

In [5]:
# New depth parameters here (replace example values with actual values)
depth_parameters = {
    'water_depth_from_mooring_diagram': 4450,  # Replace with actual value
    'water_depth_from_recorder_reading': 4562.2,      # Replace with actual value
    'corrected_water_depth': 4538.97,            # Replace with actual value
    'instrument_depth_from_mooring_diagram': -999,  # Replace with actual value
    'instrument_depth_from_mooring_log': -999,      # Replace with actual value
    'instrument_height_above_bottom': 35       # Discuss with Tom how to calcualte the height!!!
}

In [6]:
sbe16_metadata
with open("stratus12_sbe16_1876.json", "w") as output_file:
    json.dump(depth_parameters, output_file)

## Meta data: we will add depth paramter for deep T/S

In [7]:
def mat_to_xarray(mat_data, metadata):
    ds = xr.Dataset()

    # Update keys based on the actual keys in mat_data
    for param in ['temp', 'sal', 'depth']:  # Updated variable keys
        ds[param] = xr.DataArray(
            data=mat_data[param],
            dims=['time', 'depth'],
            attrs={'units': metadata['variables'][param]['attrs']['units']}
        )

    # Assign other variables
    for depth_param, value in metadata['depth_parameters'].items():
        ds[depth_param] = value

    # Assign coordinates and attributes
    for var_name, var_info in metadata['variables'].items():
        ds[var_name].attrs = var_info['attrs']
    ds.coords = metadata['coordinates']
    ds.attrs = metadata['attributes']

    return ds

In [8]:
original_json_path = '../../data/metadata/OS_NTAS_2016_D_TS.json'
new_json_path = '../../data/metadata/stratus.json'
save_preserved_info_from_json(original_json_path, new_json_path)

In [9]:
with open("stratus.json", "w") as output_file:
    json.dump(depth_parameters, output_file)
metadata_filepath = '../../data/metadata/stratus.json'
metadata = read_metadata_json(metadata_filepath)
# Inspect metadata contents
metadata

{'metadata': {'data_type': 'TS',
  'Conventions': 'CF-1.8, OceanSITES-1.4, ACDD-1.3',
  'Metadata_Conventions': 'Unidata Dataset Discovery v1.0',
  'netcdf_version': 3.6,
  'format_version': 1.4,
  'institution': 'Woods Hole Oceanographic Institution',
  'source': 'moored surface buoy',
  'naming_authority': 'OceanSITES',
  'cdm_data_type': 'Station',
  'data_assembly_center': 'Upper Ocean Processes Group at Woods Hole Oceanographic',
  'distribution_statement': 'Follows CLIVAR (Climate Variability and Predictability) standards, cf. http://www.clivar.org/resources/data/data-policy. Data available free of charge. User assumes all risk for use of data. User must display citation in any publication or product using data. User must contact PI prior to any commercial use of data.',
  'license': 'Follows CLIVAR (Climate Variability and Predictability) standards, cf. http://www.clivar.org/resources/data/data-policy. Data available free of charge. User assumes all risk for use of data. User mu

In [10]:
sbe16_metadata

array(('Stratus', '12', 'Stratus Ocean Reference Station', 'Robert Weller', array(('surface mooring', 2012, '27-May-2012 22:03:04', '03-Mar-2014 12:54:00', 735016.9791666666, 735624.5833333334, array([735016.9187963 , 735666.45549769]), 607.6645370370243, 644.6187037036289, 4539, 65, 3.7, 'Melville MV12-07', 'Ron Brown RB14-01', array((6.97132, -6.79155, 'NOAA > NESDIS > NGDC', 'http://www.ngdc.noaa.gov/geomag-web/#declination', 'IGRF11', '27-Mar-2013'),
      dtype=[('value_to_be_applied', 'O'), ('changing_by_MinutesPerYear', 'O'), ('source', 'O'), ('URL', 'O'), ('model', 'O'), ('midpoint_date', 'O')])),
      dtype=[('type', 'O'), ('start_year', 'O'), ('anchor_over_time', 'O'), ('buoy_recovery_time', 'O'), ('data_start', 'O'), ('adrift_time', 'O'), ('anchor_times', 'O'), ('days_on_station', 'O'), ('duration', 'O'), ('water_depth_m', 'O'), ('deck_height_cm', 'O'), ('watch_circle_nm', 'O'), ('deploymentcruise', 'O'), ('recoverycruise', 'O'), ('magvar', 'O')]), array(('WHOI', 'WHOI-UOP'

In [11]:
# Now call the mat_to_xarray() function
ds = mat_to_xarray(mat_data)

TypeError: mat_to_xarray() missing 1 required positional argument: 'metadata'

In [12]:
mat_to_xarray?

Signature: mat_to_xarray(mat_data, metadata)
Docstring: <no docstring>
File:      /var/folders/r_/4wqz1b4d6gjcrjk4y_7sfw9w0000h1/T/ipykernel_10121/803244816.py
Type:      function

In [ ]:
ds = mat_to_xarray(mat_data)
ds
# Verify that the DataSet is created correctly
# Check ds's content, dimensions, and attributes based on your requirements

In [ ]:
output_filepath = '../data/sample_output.nc'
# Replace with your actual file path